In [43]:
using Pkg
Pkg.activate(".")

  Activating project at `~/Desktop/Waterloo/Masters/LOO_PIT_Manual/Field_Level_Data`


In [44]:
Pkg.instantiate()
Pkg.resolve()

  No Changes to `~/Desktop/Waterloo/Masters/LOO_PIT_Manual/Field_Level_Data/Project.toml`
  No Changes to `~/Desktop/Waterloo/Masters/LOO_PIT_Manual/Field_Level_Data/Manifest.toml`


In [45]:
# Pkg.add("Turing")
# Pkg.add("Capse")
# Pkg.add("NPZ")
# Pkg.add(url="https://github.com/JuliaCosmologicalLikelihoods/PlanckLite.jl")
# Pkg.add("BenchmarkTools")
# Pkg.add("Plots")
# Pkg.add("Optim")
# Pkg.add("ForwardDiff")
# Pkg.add("LinearAlgebra")
# Pkg.add("Pathfinder")
# Pkg.add("MicroCanonicalHMC")
# Pkg.add("Transducers")
# Pkg.add("StatsPlots")
# Pkg.add("PairPlots")
# Pkg.add("LaTeXStrings")
# Pkg.add("MCMCChains")
# Pkg.add("PSIS")

In [46]:
# using Turing
# using Capse
using NPZ
# # using PlanckLite
# using BenchmarkTools
# using Plots
# using Optim
# using ForwardDiff
# #using LinearAlgebra
# using Pathfinder
# #using MicroCanonicalHMC
# #using Transducers
# using StatsPlots
# #using PairPlots
# #using LaTeXStrings
# #using CairoMakie
# using MCMCChains
using PSIS
# include("utils.jl");

Begin by running PSIS on the full voxels, and storing the results

In [47]:
full_voxel_data = Array{Vector{Float64}}(undef, 32, 32, 32)

for i in 1:32, j in 1:32, k in 1:32
    full_voxel_data[i,j,k] = []
end

In [48]:
for file in 1:8

    x = npzread("/Users/ethansmith/Desktop/Waterloo/Masters/LOO-PIT/loop_it_in_the_field/lpt_32_eh1_ovsamp1/test_with_delta_logdensity_$(file).npz")
    delta_logdensities = x["delta_logdensity"]

    for i in 1:32, j in 1:32, k in 1:32
        full_voxel_data[i, j, k] = append!(full_voxel_data[i, j, k], vec(delta_logdensities[:, :, i, j, k]))
    end
    
end

In [49]:
psis_result = Array{PSISResult{Float64, Vector{Float64}, Int64, Int64, PSIS.GeneralizedPareto{Float64}}}(undef, 32, 32, 32);
for i in 1:32, j in 1:32, k in 1:32
    psis_result[i, j, k] = psis(-full_voxel_data[i,j,k])
end

In [50]:
psis_weights = Array{Vector{Float64}}(undef, 32, 32, 32)
for i in 1:32, j in 1:32, k in 1:32
    psis_weights[i,j,k] = psis_result[i,j,k].weights
end

In [51]:
pareto_shapes = []
for i in 1:32, j in 1:32, k in 1:32
    pareto_shapes = append!(pareto_shapes, psis_result[i,j,k].pareto_shape)
end

Next, use the data to generate Inference Data object

In [52]:
# Pkg.add("ArviZ")
# Pkg.add("ArviZPythonPlots")
# Pkg.add("LinearAlgebra")
# Pkg.add("IrrationalConstants")
# Pkg.add("ArviZExampleData")
# Pkg.add("DimensionalData")
# Pkg.add("Statistics")

In [53]:
using ArviZ
using ArviZPythonPlots
using LinearAlgebra
using IrrationalConstants
using ArviZExampleData
using DimensionalData
using Statistics

In [54]:
idata = InferenceData(; log_densities=namedtuple_to_dataset((; D=full_voxel_data)))

InferenceData with groups:
  > log_densities

In [55]:
idata.log_densities[5,2,17][:D]

8192-element Vector{Float64}:
 7.091101736942965
 4.050110990270539
 3.920632926544473
 4.193935622998795
 4.0754083083278045
 4.143366632495988
 4.10152558324674
 6.554105289437507
 4.764033354591097
 4.2134348848729335
 ⋮
 3.961974708895575
 4.422106042117924
 4.15802761712735
 3.90054043022154
 4.426237167611316
 4.167237114110975
 4.177545560750063
 4.789301300716017
 4.084419328128413

Single Draw (testing)

In [56]:
first_element_all_voxels = Vector{Float64}(undef, 32768)

iterator = 1

for i in 1:32
    for j in 1:32
        for k in 1:32
            first_element_all_voxels[iterator] = full_voxel_data[i,j,k][1]
            iterator += 1
        end
    end
end

In [57]:
first_element_all_voxels

32768-element Vector{Float64}:
 4.0696051981190715
 5.4171992675443335
 4.740511732058147
 3.9991714035483463
 5.37952291104228
 4.157540506165735
 4.205445534571416
 4.2836728554292245
 4.8213511688980235
 3.9604017891844445
 ⋮
 4.382361255128817
 4.218577009602887
 4.30792764169778
 4.706306344689559
 4.1689084359898
 4.00948062945153
 3.9601031831052413
 4.067309625529965
 4.280727263188781

In [119]:
# Pkg.add("Distributions")
using Distributions

d = MvNormal(first_element_all_voxels, 4.2 * exp(2) * I)

IsoNormal(
dim: 32768
μ: [4.0696051981190715, 5.4171992675443335, 4.740511732058147, 3.9991714035483463, 5.37952291104228, 4.157540506165735, 4.205445534571416, 4.2836728554292245, 4.8213511688980235, 3.9604017891844445  …  5.557812069249611, 4.382361255128817, 4.218577009602887, 4.30792764169778, 4.706306344689559, 4.1689084359898, 4.00948062945153, 3.9601031831052413, 4.067309625529965, 4.280727263188781]
Σ: [31.03403561550873 0.0 … 0.0 0.0; 0.0 31.03403561550873 … 0.0 0.0; … ; 0.0 0.0 … 31.03403561550873 0.0; 0.0 0.0 … 0.0 31.03403561550873]
)


In [124]:
first_draw = rand(d)

32768-element Vector{Float64}:
 -1.719142369959453
 15.979983488983104
  4.9001893867976385
  1.244351071145121
  6.672836477076546
 -3.7233423401167975
  7.994887650173419
  6.400520425793764
  1.455386451105464
  4.128661440071485
  ⋮
  2.9164208195681622
  9.281704391859314
  6.83118272617115
 -4.85809315196821
 -1.5077611941499631
  2.5644951192637597
 -0.47162210678907357
  4.8317964985493225
 19.241974787987278

All draws (8192 steps of the chain, 32 x 32 x 32 voxels)

In [60]:
draws = Vector{Array{Float64}}(undef, 8192)

for n in 1:8192
    new_µ_all_voxels = Vector{Float64}(undef, 32768)

    iterator = 1
    for i in 1:32
        for j in 1:32
            for k in 1:32
                new_µ_all_voxels[iterator] = full_voxel_data[i,j,k][n]
                iterator += 1
            end
        end
    end
    dist = MvNormal(new_µ_all_voxels, I)
    draws[n] = rand(dist)
end

In [61]:
draws[1]

32768-element Vector{Float64}:
 3.7208365565379475
 5.562552855117571
 4.240657984640513
 2.610156201219314
 5.402355781623095
 4.593276841152439
 3.340140827821602
 4.23458670928004
 5.25792937650668
 4.52875531039387
 ⋮
 4.063765604418639
 3.6121641639694175
 2.792405044861846
 5.351605432684926
 4.702620234164273
 2.243090926527764
 3.3329266690231787
 3.568860689525249
 4.499110947360317

Reconstructing the draws back into a 32 x 32 x 32 array of voxels

In [62]:
draws_voxels = Vector{Array{Float64}}(undef, 8192)

for n in 1:8192
    array_draws = Array{Float64}(undef, 32, 32, 32)
    for i in 1:32
        for j in 1:32
            for k in 1:32
                array_draws[i,j,k] = draws[n][(k-1)*9 + (j-1)*3 + i]
            end
        end
    end
    draws_voxels[n] = array_draws
end

In [67]:
draws_voxels[1]

32×32×32 Array{Float64, 3}:
[:, :, 1] =
 3.72084  2.61016  3.34014  4.52876  …  5.07262  5.92147  4.64304  3.63097
 5.56255  5.40236  4.23459  3.82255     7.9376   4.72258  2.5085   4.50725
 4.24066  4.59328  5.25793  4.05551     5.50684  6.24325  4.044    2.64005
 2.61016  3.34014  4.52876  3.58326     5.92147  4.64304  3.63097  4.49691
 5.40236  4.23459  3.82255  3.37559     4.72258  2.5085   4.50725  3.4707
 4.59328  5.25793  4.05551  4.54255  …  6.24325  4.044    2.64005  3.34937
 3.34014  4.52876  3.58326  5.19886     4.64304  3.63097  4.49691  6.01515
 4.23459  3.82255  3.37559  4.8921      2.5085   4.50725  3.4707   3.22048
 5.25793  4.05551  4.54255  4.48805     4.044    2.64005  3.34937  3.80798
 4.52876  3.58326  5.19886  3.02997     3.63097  4.49691  6.01515  3.52648
 ⋮                                   ⋱                    ⋮        
 3.30203  2.95867  4.68761  2.6832      4.22496  5.13271  5.4932   4.56455
 5.86491  5.81386  3.61311  4.25617     8.31108  4.55717  3.8339   4

### Comparing against the "truth" (observations)

First, we do it for the first draw only to establish the approach.

In [107]:
truth = npzread("/Users/ethansmith/Desktop/Waterloo/Masters/LOO-PIT/loop_it_in_the_field/lpt_32_eh1_ovsamp1/truth.npz")

Dict{String, Any} with 29 entries:
  "sigma_delta_" => 0.0
  "sigma8"       => 0.807635
  "Omega_m_"     => 0.26721
  "bs2_"         => 0.0
  "b2"           => 0.0
  "bn2"          => 0.0
  "alpha_ap_"    => -0.0
  "ngbars_"      => [-0.0]
  "init_mesh_"   => [0.409135 0.603618 … 1.43194 3.28066; 0.342682 7.70378 … -3…
  "alpha_iso"    => 1.0
  "sigma_delta"  => 1.0
  "Omega_m"      => 0.313772
  "init_mesh"    => ComplexF64[0.0+0.0im 3.86137+3.17819im … 26.6917-43.6299im …
  "b1"           => 1.1
  "bnp"          => 0.0
  "ngbars"       => [0.000843318]
  "fNL"          => 0.0
  "obs"          => [132.097 103.195 … 259.971 108.95; 109.023 250.692 … 323.32…
  "sigma_0_"     => -0.0
  ⋮              => ⋮

In [110]:
obs = truth["obs"]

32×32×32 Array{Float64, 3}:
[:, :, 1] =
 132.097   103.195  169.831   …  312.164   140.722  259.971  108.95
 109.023   250.692  134.742      230.23    161.214  323.321  272.411
 302.156   194.979  209.445      264.093   172.311  218.114  243.39
 212.498   188.725  179.956       99.7786  235.612  107.006  262.533
 215.067   195.694  172.687      125.386   159.716  206.637  312.8
 125.941   386.437  202.351   …  240.082   107.686  120.323  200.874
 145.982   278.818  219.986      262.611   221.983  111.508  181.234
  57.9255  265.68   165.435      309.174   325.274  167.692  268.522
 235.33    295.878  221.743      255.399   165.987  219.4    221.629
  28.5709  123.829  134.757      156.588   230.92   241.889  155.009
   ⋮                          ⋱                       ⋮      
 161.77    251.528  278.364      211.109   139.991  267.002  169.525
 289.303   197.795   59.1787     345.762   380.446  287.012  210.888
 247.471   334.325   99.3467  …  219.044   248.65   163.439  204.857
 194.

In [127]:
first_draw_array = Array{Float64}(undef, 32, 32, 32)
iterator = 1
for i in 1:32
    for j in 1:32
        for k in 1:32
            first_draw_array[i,j,k] = first_draw[(k-1)*9 + (j-1)*3 + i]
            iterator += 1
        end
    end
end

In [134]:
values = Vector{Float64}(undef, 32768)
iterator = 1
for i in 1:32
    for j in 1:32
        for k in 1:32
            if first_draw_array[i,j,k] > obs[i,j,k]
                values[iterator] = 1
            else
                values[iterator] = 0
            end
            iterator += 1
        end
    end
end
sum(values)/length(values)

0.000732421875

Now for all draws.

In [136]:
obs = truth["obs"]

32×32×32 Array{Float64, 3}:
[:, :, 1] =
 132.097   103.195  169.831   …  312.164   140.722  259.971  108.95
 109.023   250.692  134.742      230.23    161.214  323.321  272.411
 302.156   194.979  209.445      264.093   172.311  218.114  243.39
 212.498   188.725  179.956       99.7786  235.612  107.006  262.533
 215.067   195.694  172.687      125.386   159.716  206.637  312.8
 125.941   386.437  202.351   …  240.082   107.686  120.323  200.874
 145.982   278.818  219.986      262.611   221.983  111.508  181.234
  57.9255  265.68   165.435      309.174   325.274  167.692  268.522
 235.33    295.878  221.743      255.399   165.987  219.4    221.629
  28.5709  123.829  134.757      156.588   230.92   241.889  155.009
   ⋮                          ⋱                       ⋮      
 161.77    251.528  278.364      211.109   139.991  267.002  169.525
 289.303   197.795   59.1787     345.762   380.446  287.012  210.888
 247.471   334.325   99.3467  …  219.044   248.65   163.439  204.857
 194.

In [137]:
draws_voxels[1]

32×32×32 Array{Float64, 3}:
[:, :, 1] =
 3.72084  2.61016  3.34014  4.52876  …  5.07262  5.92147  4.64304  3.63097
 5.56255  5.40236  4.23459  3.82255     7.9376   4.72258  2.5085   4.50725
 4.24066  4.59328  5.25793  4.05551     5.50684  6.24325  4.044    2.64005
 2.61016  3.34014  4.52876  3.58326     5.92147  4.64304  3.63097  4.49691
 5.40236  4.23459  3.82255  3.37559     4.72258  2.5085   4.50725  3.4707
 4.59328  5.25793  4.05551  4.54255  …  6.24325  4.044    2.64005  3.34937
 3.34014  4.52876  3.58326  5.19886     4.64304  3.63097  4.49691  6.01515
 4.23459  3.82255  3.37559  4.8921      2.5085   4.50725  3.4707   3.22048
 5.25793  4.05551  4.54255  4.48805     4.044    2.64005  3.34937  3.80798
 4.52876  3.58326  5.19886  3.02997     3.63097  4.49691  6.01515  3.52648
 ⋮                                   ⋱                    ⋮        
 3.30203  2.95867  4.68761  2.6832      4.22496  5.13271  5.4932   4.56455
 5.86491  5.81386  3.61311  4.25617     8.31108  4.55717  3.8339   4

In [147]:
values = Vector{Float64}(undef, 32768)
iterator = 1
for i in 1:32
    for j in 1:32
        for k in 1:32
            if draws_voxels[1][i,j,k] > obs[i,j,k]
                values[iterator] = 1
            else
                values[iterator] = 0
            end
            iterator += 1
        end
    end
end
sum(values)/length(values)

0.000762939453125